In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
if os.path.exists('vendas.csv'):
    df = pd.read_csv('vendas.csv')
else:
    np.random.seed(42)
    n = 300
    lojas = ['Loja Centro', 'Loja Norte', 'Loja Sul', 'Loja Leste']
    regiao_da_loja = {'Loja Centro': 'Sudeste', 'Loja Norte': 'Norte',
                      'Loja Sul': 'Sul', 'Loja Leste': 'Nordeste'}
    produtos = {  
        'Notebook': ('Eletrônicos', 3000), 'Celular': ('Eletrônicos', 1800),
        'Mouse': ('Acessórios', 60), 'Teclado': ('Acessórios', 120),
        'Monitor': ('Eletrônicos', 900), 'Cadeira': ('Móveis', 700),
        'Mesa': ('Móveis', 550), 'Headset': ('Acessórios', 200),
    }
    nomes = list(produtos.keys())
    df = pd.DataFrame({
        'data': pd.date_range('2024-01-01', '2024-12-31', periods=n).astype(str),
        'loja': np.random.choice(lojas, n),
        'produto': np.random.choice(nomes, n),
        'quantidade': np.random.randint(1, 10, n).astype(float),
    })
    df['preco_unitario'] = df['produto'].map(lambda p: produtos[p][1]) * np.random.uniform(0.9, 1.1, n)
    df['categoria'] = df['produto'].map(lambda p: produtos[p][0])
    df['regiao'] = df['loja'].map(regiao_da_loja)
    
    df.loc[np.random.choice(n, 10, replace=False), 'preco_unitario'] = np.nan
    df.loc[np.random.choice(n, 10, replace=False), 'quantidade'] = np.nan


df['data'] = pd.to_datetime(df['data'])


if 'categoria' not in df.columns:
    df['categoria'] = df['produto']
if 'regiao' not in df.columns:
    df['regiao'] = df['loja']

In [ ]:
df['faturamento'] = df['quantidade'] * df['preco_unitario']

In [ ]:
print("=== 4) df.info() ===")
df.info()
print("\n=== Valores ausentes ===")
print(df.isnull().sum())
print("\n=== Resumo estatístico ===")
print(df.describe())


In [ ]:
df['preco_unitario'] = df['preco_unitario'].fillna(df['preco_unitario'].mean())
df['quantidade'] = df['quantidade'].fillna(df['quantidade'].mean())
# Recalcula o faturamento após o tratamento
df['faturamento'] = df['quantidade'] * df['preco_unitario']

In [ ]:
fat_loja = df.groupby('loja')['faturamento'].sum().sort_values(ascending=False)
print("\n=== 6) Faturamento por loja ===")
print(fat_loja)

In [ ]:
ticket_medio = df.groupby('loja')['faturamento'].sum() / df.groupby('loja')['faturamento'].count()
print("\n=== 7) Ticket médio por loja ===")
print(ticket_medio)

In [ ]:
top5 = df.groupby('produto')['quantidade'].sum().sort_values(ascending=False).head(5)
print("\n=== 8) Top 5 produtos mais vendidos ===")
print(top5)


In [ ]:
df['mes_ano'] = df['data'].dt.to_period('M')
fat_mensal = df.groupby('mes_ano')['faturamento'].sum()
mes_maior = fat_mensal.idxmax().strftime('%m/%Y')
mes_menor = fat_mensal.idxmin().strftime('%m/%Y')
print("\n=== 9) Meses extremos ===")
print(f"Maior faturamento: {mes_maior} (R$ {fat_mensal.max():,.2f})")
print(f"Menor faturamento: {mes_menor} (R$ {fat_mensal.min():,.2f})")



In [ ]:
df['mes'] = df['data'].dt.month

In [ ]:
plt.figure(figsize=(8, 5))
cores = sns.color_palette('deep', len(fat_loja))
plt.bar(fat_loja.index, fat_loja.values, color=cores, label='Faturamento total')
plt.title('Faturamento Total por Loja')
plt.xlabel('Loja')
plt.ylabel('Faturamento (R$)')
plt.legend()
plt.tight_layout()
plt.savefig('faturamento_por_loja.png', dpi=150)
plt.savefig('graficos/faturamento_por_loja.png', dpi=150)
plt.show()


In [ ]:
mensal = fat_mensal.reset_index()
mensal['mes_ano'] = mensal['mes_ano'].dt.strftime('%m/%Y')
plt.figure(figsize=(10, 5))
plt.plot(mensal['mes_ano'], mensal['faturamento'], marker='o', label='Faturamento mensal')
for x, y in zip(mensal['mes_ano'], mensal['faturamento']):
    plt.text(x, y, f'{y:,.0f}', ha='center', va='bottom', fontsize=8)
plt.title('Evolução Mensal do Faturamento')
plt.xlabel('Mês/Ano')
plt.ylabel('Faturamento (R$)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('graficos/evolucao_mensal.png', dpi=150)
plt.show()

In [ ]:
fat_regiao = df.groupby('regiao')['faturamento'].sum()
plt.figure(figsize=(7, 7))
plt.pie(fat_regiao.values, labels=fat_regiao.index, autopct='%1.1f%%',
        startangle=90, colors=sns.color_palette('deep', len(fat_regiao)))
plt.title('Distribuição do Faturamento por Região')
plt.tight_layout()
plt.savefig('graficos/faturamento_por_regiao.png', dpi=150)
plt.show()



In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='quantidade', y='faturamento', hue='categoria')
plt.title('Relação entre Quantidade Vendida e Faturamento')
plt.xlabel('Quantidade de itens vendidos')
plt.ylabel('Faturamento (R$)')
plt.tight_layout()
plt.savefig('graficos/dispersao_qtd_faturamento.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='categoria', y='preco_unitario', hue='categoria', legend=False)
plt.title('Distribuição de Preços Unitários por Categoria')
plt.xlabel('Categoria')
plt.ylabel('Preço unitário (R$)')
plt.tight_layout()
plt.savefig('graficos/boxplot_precos.png', dpi=150)
plt.show()


In [ ]:
corr = df[['quantidade', 'preco_unitario', 'faturamento', 'mes']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlação entre Variáveis Numéricas')
plt.tight_layout()
plt.savefig('graficos/heatmap_correlacao.png', dpi=150)
plt.show()



In [ ]:
sns.set_style('whitegrid')
sns.set_palette(sns.color_palette('deep'))

os.makedirs('graficos', exist_ok=True)



In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.bar(fat_loja.index, fat_loja.values, color=sns.color_palette('deep', len(fat_loja)))
plt.title('Faturamento por Loja')
plt.xlabel('Loja')
plt.ylabel('Faturamento (R$)')
plt.xticks(rotation=30)

plt.subplot(1, 2, 2)
plt.bar(fat_regiao.index, fat_regiao.values, color=sns.color_palette('deep', len(fat_regiao)))
plt.title('Faturamento por Região')
plt.xlabel('Região')
plt.ylabel('Faturamento (R$)')

plt.suptitle('Comparativo de Faturamento: Loja x Região')
plt.tight_layout()
plt.savefig('graficos/comparativo_loja_regiao.png', dpi=150)
plt.show()


In [ ]:
resumo = pd.DataFrame({
    'faturamento_total': fat_loja,
    'ticket_medio': ticket_medio,
}).reset_index().rename(columns={'index': 'loja'})
resumo['mes_maior_faturamento'] = mes_maior
resumo['mes_menor_faturamento'] = mes_menor
resumo['top5_produtos'] = ', '.join(top5.index)
resumo.to_csv('resumo_analitico.csv', index=False, encoding='utf-8-sig')
print("\n=== 20) resumo_analitico.csv ===")
print(resumo)